# Fine-tuning YOLO11-OBB on MAR20 (military aircraft, 20 types)

**Kaggle notebook settings:** Accelerator: GPU T4 x2 or P100; Internet: On (requires a verified phone number).

**Data:** Add Input -> the full MAR20 dataset (3842 images; folders `Annotations/Oriented Bounding Boxes`, `ImageSets/Main`, `JPEGImages`).
The path is detected automatically. Official archive: https://gcheng-nwpu.github.io/ (CC BY-NC 4.0);
upload it as your own Kaggle Dataset if needed. The Hugging Face mirror has only 100 images and is not usable.

**Background run:** Save Version -> Save & Run All. Weights, metrics and plots end up in the version Output (`/kaggle/working/export`).

Training starts from `yolo11s-obb.pt` (Ultralytics, DOTA-v1.0, AGPL-3.0): transfer learning.

In [ ]:
REPO_URL = "https://github.com/miss-mississippi/muxxed_defence_tech.git"
MAR20_SRC = None          # None: find MAR20 in /kaggle/input automatically
MODEL = "yolo11s-obb.pt"  # yolo11m-obb.pt is more accurate but ~2.8x slower
EPOCHS, IMGSZ, BATCH = 60, 800, 16  # MAR20 images are about 800x800
NOISE = 0.0               # 0.5 adds Gaussian noise to augmentations (fixes the noise weakness)
RUN_NAME = "mar20_s_800"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf /kaggle/working/repo && git clone -q $REPO_URL /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q ultralytics==8.4.152

In [ ]:
import json
from pathlib import Path

if MAR20_SRC is None:
    hits = sorted(Path("/kaggle/input").glob("**/Annotations/Oriented Bounding Boxes"))
    if not hits:
        raise SystemExit("MAR20 not found in /kaggle/input: Add Input -> the full MAR20 dataset")
    MAR20_SRC = str(hits[0].parent.parent)
print("MAR20:", MAR20_SRC)

!python scripts/prepare_mar20.py --src "$MAR20_SRC" --out /kaggle/working/mar20_yolo

stats = json.loads(Path("/kaggle/working/mar20_yolo/stats.json").read_text())
total = sum(stats["images"].values())
assert total > 3000, f"the dataset has {total} images, expected 3842: check the source"

In [ ]:
import torch
DEVICE = ",".join(str(i) for i in range(torch.cuda.device_count())) or "cpu"
print("device:", DEVICE, "| noise:", NOISE)
!python scripts/train.py --data /kaggle/working/mar20_yolo/mar20.yaml --weights $MODEL --name $RUN_NAME \
    --epochs $EPOCHS --imgsz $IMGSZ --batch $BATCH --device $DEVICE --workers 4 --noise $NOISE

In [ ]:
import glob, shutil
from IPython.display import Image, Markdown, display

export = Path("/kaggle/working/export")
export.mkdir(exist_ok=True)
run_dir = Path("runs/obb") / RUN_NAME
files = [f"weights/{RUN_NAME}.pt", *glob.glob(f"outputs/eval/{RUN_NAME}_*.*"), run_dir / "results.csv", *run_dir.glob("*.png")]
files += glob.glob("outputs/robustness/robustness.*")
for f in files:
    shutil.copy2(f, export)
for md in [*glob.glob(f"outputs/eval/{RUN_NAME}_*.md"), "outputs/robustness/robustness.md"]:
    if Path(md).exists():
        display(Markdown(Path(md).read_text()))
display(Image(str(run_dir / "results.png")))
print(sorted(p.name for p in export.iterdir()))